# 14 · Final holdout evaluation

**One pass over the untouched holdout:** 3,022 households in 336 clusters, and 622 high-need households for the VUP model. Every model compared here was fixed from development results (notebook 13) **before this notebook ran**. No tuning, selection, threshold or calibration step uses holdout labels.

**Models compared.**
- **Poverty model:**
  - the district poverty-rate baseline;
  - the **best unmitigated model** (notebook 13's selection rule applied to candidates without recalibration);
  - the **best group-recalibrated model** (the same rule applied to recalibrated candidates);
  - as a supplement, the recalibrated model's **own base model before recalibration**, which isolates the effect of recalibration.
- **VUP coverage-gap model:** the deployment model (and its version before recalibration, if it was recalibrated) against the district coverage baseline.
- **Deployment model:** notebook 13's overall selection, saved as `models/*_final.joblib`. The holdout results below are reported, **not used to change that choice**.

**Prior holdout use, disclosed.** Notebooks 09–11 scored the *default* models on this holdout. Nothing was selected from those scores. All models compared here are new to the holdout.

| | |
|---|---|
| **Uncertainty** | Cluster bootstrap, 1,000 draws, clusters resampled within strata. Differences are paired. |
| **Weights** | `weight` for household metrics. "Poor people reached" uses `pop_wt`. |
| **Outputs** | `reports/final_model_metrics.csv`, `reports/final_evaluation/*.csv`, `reports/figures/final_evaluation/*.png` |

In [ ]:
import json
import sys
import warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "models" / "common.py").exists())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import precision_recall_curve, roc_curve
from sklearn.pipeline import Pipeline

from src.models import common as cm
from src.models import fairness as fr
from src.models import plots as pl
from src.models import poverty_model as pm
from src.models import training as tr
from src.models import tuning as tu
from src.models import vup_gap_model as vm

pd.set_option("display.width", 230)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.3f}".format)
FIG = ROOT / "reports" / "figures" / "final_evaluation"
OUT = ROOT / "reports" / "final_evaluation"
FIG.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

pov_path, vup_path = cm.MODELS_DIR / "poverty_vulnerability_pipeline_final.joblib", cm.MODELS_DIR / "vup_gap_pipeline_final.joblib"
pov_final, vup_final = joblib.load(pov_path), joblib.load(vup_path)
pov_meta, vup_meta = json.loads(pov_path.with_suffix(".json").read_text()), json.loads(vup_path.with_suffix(".json").read_text())
results = pd.read_csv(ROOT / "reports" / "tuning_results.csv")
results["params"] = results["params"].apply(json.loads)
tol = json.loads((ROOT / "reports" / "tuning_gate_tolerances.json").read_text())

df = cm.load_frame().merge(cm.load_splits(), on=["hhid", "clust"])
dev = df[df["split"].eq("development")].reset_index(drop=True)
hold = fr.add_groups(df[df["split"].eq("holdout")].reset_index(drop=True))
spec = pm.FEATURE_SETS["full"]
print(f"Deployment models (notebook 13): poverty = {pov_meta['model']}; VUP = {vup_meta['model']}")

## 1. Poverty model: which models are compared (development results only)

In [ ]:
pov_res = results[results["model_target"].eq("poverty")].drop(columns=["gate_performance", "gate_fairness", "gate_calibration", "survivor", "selected"])
unm_name, unm_tab, _ = tu.select_final(pov_res[~pov_res["recalibrated"]], "pr_auc", tol["poverty"])
rec_name, rec_tab, _ = tu.select_final(pov_res[pov_res["recalibrated"]], "pr_auc", tol["poverty"])
deploy = pov_meta["model"]
assert deploy in (unm_name, rec_name), "deployment model must be one of the two compared models"
print(f"Best unmitigated (rule on unmitigated candidates): {unm_name}")
print(f"Best group-recalibrated (rule on recalibrated candidates): {rec_name}")
print(f"Deployment (overall rule, notebook 13): {deploy}")
dev_view = pov_res.set_index("candidate").loc[[unm_name, rec_name], ["pr_auc_mean", "pr_auc_se", "roc_auc", "recall_people_top20", "calibration_slope",
                                                                    "recall_gap", "worst_group_recall", "mean_abs_calibration_gap", "max_abs_calibration_gap", "complexity"]]
print("Development (out-of-fold) results that fixed these choices:")
dev_view

In [ ]:
def fit_candidate(row, data, feature_spec, target, cal_groups):
    """Fit a candidate on the development data: base pipeline, plus a calibrator fitted on cross-fitted out-of-fold scores."""
    params = row["params"] or None
    base = cm.fit_pipeline(tu.pipeline(feature_spec, row["model"], params), data[feature_spec.columns], data[target], data["weight"])
    if not row["recalibrated"]:
        return base
    oof = fr.cv_predict(tu.pipeline(feature_spec, row["model"], params), data, feature_spec.columns, target, data["fold"])
    cal = fr.GroupCalibrator(cal_groups).fit(oof, fr.add_groups(data), data[target], data["weight"])
    return fr.GroupRecalibratedClassifier(base, cal)

rows_by = pov_res.set_index("candidate")
models = {}
for role, name in [("unmitigated", unm_name), ("recalibrated", rec_name)]:
    models[role] = pov_final if name == deploy else fit_candidate(rows_by.loc[name], dev, spec, "poor", fr.CALIBRATION_GROUPS)
base_pov = cm.fit_pipeline(Pipeline([("model", cm.GroupRateBaseline("district"))]), dev[["district"]], dev["poor"], dev["weight"])
LABEL = {"district_baseline": "District baseline", "unmitigated": f"Unmitigated: {unm_name}", "recalibrated": f"Group-recalibrated: {rec_name}",
         "recalibrated_base": f"Supplementary: {rec_name.replace('+group_recal', '')} before recalibration"}
print({k: type(v).__name__ for k, v in models.items()})

## 2. Poverty model: the holdout pass

In [ ]:
X = hold[spec.columns]
hold["district_baseline"] = base_pov.predict_proba(hold[["district"]])[:, 1]
hold["unmitigated"] = models["unmitigated"].predict_proba(X)[:, 1]
hold["recalibrated"] = models["recalibrated"].predict_proba(X)[:, 1]
hold["recalibrated_base"] = models["recalibrated"].base.predict_proba(X)[:, 1]
SCORES = ["district_baseline", "unmitigated", "recalibrated_base", "recalibrated"]
print("Supplementary column 'recalibrated_base' = the recalibrated pipeline's own base model, to isolate the effect of recalibration.")
METRICS = ["roc_auc", "pr_auc", "brier", "log_loss", "precision_top10", "recall_top10", "recall_people_top10", "precision_top20", "recall_top20",
           "recall_people_top20", "precision_top30", "recall_top30", "recall_people_top30", "calibration_slope", "calibration_intercept", "calibration_in_the_large"]
point = pd.DataFrame({s: cm.evaluate(hold["poor"], hold[s], hold["weight"], hold["pop_wt"]) for s in SCORES}).loc[METRICS]
BOOT_M = ("roc_auc", "pr_auc", "brier", "precision_top10", "recall_top10", "recall_people_top10", "precision_top20", "recall_top20", "recall_people_top20",
          "precision_top30", "recall_top30", "recall_people_top30", "calibration_in_the_large")
b_base = cm.cluster_bootstrap(hold, SCORES, "poor", B=1000, baseline="district_baseline", metrics=BOOT_M).set_index(["score", "metric"])
b_unm = cm.cluster_bootstrap(hold, ["unmitigated", "recalibrated"], "poor", B=1000, baseline="unmitigated", metrics=BOOT_M).set_index(["score", "metric"])
b_rec = cm.cluster_bootstrap(hold, ["recalibrated_base", "recalibrated"], "poor", B=1000, baseline="recalibrated_base", metrics=BOOT_M).set_index(["score", "metric"])
tab = point.copy()
for s in SCORES:
    tab[f"{s} CI"] = [f"{b_base.loc[(s, m), 'ci_low']:.3f}-{b_base.loc[(s, m), 'ci_high']:.3f}" if (s, m) in b_base.index else "" for m in METRICS]
for s in ["unmitigated", "recalibrated_base", "recalibrated"]:
    tab[f"{s} - baseline"] = [f"{b_base.loc[(s, m), 'diff_vs_baseline_ci_low']:+.3f} to {b_base.loc[(s, m), 'diff_vs_baseline_ci_high']:+.3f}" if (s, m) in b_base.index else "" for m in METRICS]
tab["recalibrated - unmitigated"] = [f"{b_unm.loc[('recalibrated', m), 'diff_vs_baseline_ci_low']:+.3f} to {b_unm.loc[('recalibrated', m), 'diff_vs_baseline_ci_high']:+.3f}"
                                     if ("recalibrated", m) in b_unm.index else "" for m in METRICS]
tab["recalibrated - its base"] = [f"{b_rec.loc[('recalibrated', m), 'diff_vs_baseline_ci_low']:+.3f} to {b_rec.loc[('recalibrated', m), 'diff_vs_baseline_ci_high']:+.3f}"
                                  if ("recalibrated", m) in b_rec.index else "" for m in METRICS]
tab.to_csv(OUT / "poverty_holdout_metrics.csv")
tab

In [ ]:
fig, axes = pl.figure("Poverty model on the holdout: targeting and calibration",
                      "Left: share of poor people reached for each share of households targeted (dot = current VUP). Right: calibration by decile of prediction",
                      width=12, height=5.2, ncols=2)
a1, a2 = axes
vup_alloc = tr.allocation_metrics(hold["poor"], hold["vup_any"], hold["weight"], hold["pop_wt"])
k_vup = vup_alloc["share_selected"]
COL = {"district_baseline": pl.NEUTRAL, "unmitigated": pl.SERIES[0], "recalibrated_base": pl.SERIES[1], "recalibrated": pl.SERIES[2]}
a1.plot([0, 100], [0, 100], ":", color=pl.BASELINE)
for s in SCORES:
    t = tr.targeting_curve(hold["poor"], hold[s], hold["weight"], hold["pop_wt"])
    a1.plot(100 * t["k"], 100 * t["recall_people"], "--" if s == "district_baseline" else "-", color=COL[s], linewidth=2, label=LABEL[s])
a1.plot(100 * k_vup, 100 * vup_alloc["recall_people"], "D", color=pl.SERIES[7], markersize=9, markeredgecolor=pl.SURFACE, label="Current VUP allocation")
a1.set_xlabel("Households targeted, %", fontsize=9, color=pl.INK["secondary"]); a1.set_ylabel("Poor people reached, %", fontsize=9, color=pl.INK["secondary"])
a1.legend(frameon=False, fontsize=8, loc="lower right"); a1.grid(axis="both", color=pl.GRID)
a2.plot([0, 1], [0, 1], ":", color=pl.BASELINE)
cal_tables = {}
for s in SCORES:
    c = cm.calibration_table(hold["poor"], hold[s], hold["weight"])
    cal_tables[s] = c
    a2.plot(c["mean_predicted"], c["observed"], "o-", color=COL[s], linewidth=2, markersize=4, label=LABEL[s])
a2.set_xlabel("Mean predicted probability", fontsize=9, color=pl.INK["secondary"]); a2.set_ylabel("Observed share poor", fontsize=9, color=pl.INK["secondary"])
a2.legend(frameon=False, fontsize=8); a2.grid(axis="both", color=pl.GRID)
path = pl.save(fig, FIG / "poverty_targeting_calibration.png", top=0.83, bottom=0.12, left=0.07, wspace=0.22)
display(Image(str(path)))
pd.concat(cal_tables, axis=1).to_csv(OUT / "poverty_calibration_deciles.csv")
pd.concat(cal_tables, axis=1)

In [ ]:
budget_rows = {}
for s in SCORES:
    e = cm.evaluate(hold["poor"], hold[s], hold["weight"], hold["pop_wt"], k_values=(k_vup,))
    tag = f"top{int(round(100 * k_vup))}"
    budget_rows[LABEL[s]] = {"precision": e[f"precision_{tag}"], "poor people reached": e[f"recall_people_{tag}"]}
budget_rows["Current VUP allocation"] = {"precision": vup_alloc["precision"], "poor people reached": vup_alloc["recall_people"]}
budget = pd.DataFrame(budget_rows).T
print(f"At the current VUP budget ({100 * k_vup:.1f}% of holdout households):")
budget

In [ ]:
def confusion(y, selected, w):
    y, s, w = (np.asarray(a, float) for a in (y, selected, w))
    cells = {"selected & poor (TP)": (s * y * w).sum(), "selected & not poor (FP)": (s * (1 - y) * w).sum(),
             "not selected & poor (FN)": ((1 - s) * y * w).sum(), "not selected & not poor (TN)": ((1 - s) * (1 - y) * w).sum()}
    return pd.Series(cells)

conf = {}
for s in ["unmitigated", "recalibrated"]:
    conf[(s, "top 20% of households")] = confusion(hold["poor"], cm.top_k_inclusion(hold[s], hold["weight"], 0.20), hold["weight"])
    conf[(s, "probability >= 0.5")] = confusion(hold["poor"], (hold[s] >= 0.5).astype(float), hold["weight"])
conf = pd.DataFrame(conf)
conf.to_csv(OUT / "poverty_confusion_matrices.csv")
pct = 100 * conf / conf.sum()
display(conf.round(0)); display(pct.round(1))
for col in conf.columns:
    tp, fp, fn, tn = conf[col]
    print(f"{col}: selected {100 * (tp + fp) / conf[col].sum():.1f}% | precision {tp / (tp + fp):.3f} | recall (households) {tp / (tp + fn):.3f} | "
          f"inclusion error {fp / (tp + fp):.3f} | exclusion error {fn / (tp + fn):.3f}")

### How group recalibration changes the ranking

The recalibrated model passes the base model's score *p* through one fitted logistic layer:

logit(*q*) = *a* · logit(*p*) + *b* + Σ_g *c_g* · 1[household in group *g*]

- The groups are: member with a disability (yes/no), head age (under 35, 35–59, 60+), province (5) and urban/rural.
- *a*, *b* and the *c_g* are fitted on **cross-fitted out-of-fold** development scores (weighted logistic regression). The calibrator therefore never sees a score the base model produced for its own training rows.
- Within a group the transformation is monotone (*a* > 0), so **the ranking inside each group does not change**.
- Between groups, the intercepts *c_g* move whole groups up or down. This changes **who crosses a national cut-off**, and therefore the national ranking.

The cells below show the fitted coefficients and how much the holdout ranking moves. They use the holdout's features only, no labels.

In [ ]:
cal_model = models["recalibrated"].calibrator
coef = pd.Series(cal_model.model_.coef_[0], index=cal_model.columns_)
print(f"Slope a on logit(p): {coef['logit_p']:.3f}; intercept b: {cal_model.model_.intercept_[0]:.3f}")
gc = coef.drop("logit_p").rename("log-odds shift c_g").to_frame()
gc["odds multiplier"] = np.exp(gc["log-odds shift c_g"])
gc.to_csv(OUT / "group_recalibration_coefficients.csv")
display(gc)
same_base = unm_name == rec_name.replace("+group_recal", "")
p_base = models["recalibrated"].base.predict_proba(X)[:, 1]
rho = pd.Series(p_base).rank().corr(hold["recalibrated"].rank())
inc_b = cm.top_k_inclusion(p_base, hold["weight"], 0.20)
inc_r = cm.top_k_inclusion(hold["recalibrated"], hold["weight"], 0.20)
overlap = (np.minimum(inc_b, inc_r) * hold["weight"]).sum() / (inc_b * hold["weight"]).sum()
print(f"Base model of the recalibrated pipeline {'equals' if same_base else 'differs from'} the unmitigated model.")
if same_base:
    print(f"Check: max |unmitigated score - recalibrated pipeline's base score| on the holdout = {np.abs(hold['unmitigated'].to_numpy() - p_base).max():.2e}")
print(f"Spearman rank correlation, base vs recalibrated scores (holdout): {rho:.4f}")
print(f"Share of the top-20% selection kept after recalibration: {100 * overlap:.1f}% (so {100 * (1 - overlap):.1f}% of selected households change)")
sel = pd.DataFrame({"selection rate before": pd.Series(inc_b).groupby(hold["province"]).apply(lambda s: np.average(s, weights=hold.loc[s.index, "weight"])),
                    "selection rate after": pd.Series(inc_r).groupby(hold["province"]).apply(lambda s: np.average(s, weights=hold.loc[s.index, "weight"]))})
for dim in ["disability", "head_age_group", "area"]:
    sel = pd.concat([sel, pd.DataFrame({"selection rate before": pd.Series(inc_b).groupby(hold[dim]).apply(lambda s: np.average(s, weights=hold.loc[s.index, "weight"])),
                                        "selection rate after": pd.Series(inc_r).groupby(hold[dim]).apply(lambda s: np.average(s, weights=hold.loc[s.index, "weight"]))})])
sel["change, pp"] = 100 * (sel["selection rate after"] - sel["selection rate before"])
sel.to_csv(OUT / "group_recalibration_selection_shift.csv")
sel

## 3. Poverty model: subgroup metrics (holdout)

In [ ]:
sub = {s: fr.group_table_ci(hold, s, "poor", B=500) for s in ["unmitigated", "recalibrated_base", "recalibrated"]}
sub["district_baseline"] = fr.group_table(hold, "district_baseline", "poor")
for s, t in sub.items():
    t.to_csv(OUT / f"poverty_subgroups_{s}.csv", index=False)
key = ["dimension", "group", "households", "positives", "observed_rate"]
cmp = sub["unmitigated"][key].copy()
for s in ["unmitigated", "recalibrated_base", "recalibrated"]:
    for c in ["mean_predicted", "mean_error", "roc_auc", "pr_auc", "precision_top20", "recall_people_top20", "selection_rate"]:
        cmp[f"{c} [{s}]"] = sub[s][c].to_numpy()
    cmp[f"mean_error CI [{s}]"] = [f"{lo:+.3f} to {hi:+.3f}" for lo, hi in zip(sub[s]["mean_error_ci_low"], sub[s]["mean_error_ci_high"])]
    cmp[f"recall CI [{s}]"] = [f"{lo:.2f}-{hi:.2f}" for lo, hi in zip(sub[s]["recall_people_top20_ci_low"], sub[s]["recall_people_top20_ci_high"])]
cmp["recall_people_top20 [district_baseline]"] = sub["district_baseline"]["recall_people_top20"].to_numpy()
cmp.to_csv(OUT / "poverty_subgroup_comparison.csv", index=False)
fair = pd.DataFrame({s: fr.fairness_summary(sub[s]) for s in ["district_baseline", "unmitigated", "recalibrated_base", "recalibrated"]})
fair.to_csv(OUT / "poverty_fairness_indices.csv")
display(fair)
cmp

In [ ]:
t = cmp.copy()
t.index = [f"{d.replace('_', ' ')}: {g}" for d, g in zip(t["dimension"], t["group"])]
n = len(t)
h = 1.9 + 0.27 * n
fig, (a1, a2, a3) = pl.figure("Poverty model by group on the holdout: unmitigated vs group-recalibrated",
                              "National top-20% cut-off. Blue = best unmitigated; orange = recalibrated model's base; green = group-recalibrated (95% interval). AUC: groups with >= 20 poor",
                              width=14, height=h, ncols=3)
y = np.arange(n)[::-1]
for ax, col, title, scale in [(a1, "recall_people_top20", "Poor people selected, %", 100), (a2, "mean_error", "Mean prediction error, pp", 100), (a3, "roc_auc", "ROC-AUC within group", 1)]:
    ci = sub["recalibrated"]
    lo, hi = ci[f"{col}_ci_low"] if f"{col}_ci_low" in ci else None, ci[f"{col}_ci_high"] if f"{col}_ci_high" in ci else None
    if lo is not None:
        ax.hlines(y, scale * lo.to_numpy(), scale * hi.to_numpy(), color=pl.SERIES[2], alpha=0.35, linewidth=2)
    ax.scatter(scale * t[f"{col} [unmitigated]"], y, color=pl.SERIES[0], s=28, zorder=3)
    ax.scatter(scale * t[f"{col} [recalibrated_base]"], y, color=pl.SERIES[1], s=22, zorder=3, marker="s")
    ax.scatter(scale * t[f"{col} [recalibrated]"], y, color=pl.SERIES[2], s=28, zorder=4, marker="D")
    if col == "mean_error":
        ax.axvline(0, color=pl.INK["muted"], linestyle="--", linewidth=1)
    ax.set_title(title, fontsize=9.5, color=pl.INK["secondary"], loc="left")
    ax.set_ylim(-0.7, n - 0.3)
a1.set_yticks(y, [f"{i}  (n={hh:,})" for i, hh in zip(t.index, t["households"])], fontsize=8.5)
for ax in (a2, a3):
    ax.set_yticks(y, [""] * n)
path = pl.save(fig, FIG / "poverty_subgroups_holdout.png", top=1 - 1.3 / h, bottom=0.5 / h, left=0.22, wspace=0.1)
display(Image(str(path)))

In [ ]:
# Policy option (not part of the model): separate urban/rural cut-offs on the deployment model's score
dep_col = "recalibrated" if deploy == rec_name else "unmitigated"
inc_q = fr.area_quota_inclusion(hold[dep_col], hold["weight"], hold["pop_wt"], hold["urban"])
t_q = fr.group_table(hold, dep_col, "poor", inclusion=inc_q).set_index(["dimension", "group"])
t_n = sub[dep_col].set_index(["dimension", "group"])
yv, wv, pwv = hold["poor"].to_numpy(), hold["weight"].to_numpy(), hold["pop_wt"].to_numpy()
policy = pd.DataFrame({"national cut-off": {"poor people reached": point.loc["recall_people_top20", dep_col], "precision": point.loc["precision_top20", dep_col],
                                            "urban poor reached": t_n.loc[("area", "urban"), "recall_people_top20"], "rural poor reached": t_n.loc[("area", "rural"), "recall_people_top20"]},
                       "separate urban/rural cut-offs (policy option)": {"poor people reached": (inc_q * pwv * yv).sum() / (pwv * yv).sum(),
                                                                          "precision": (inc_q * wv * yv).sum() / (inc_q * wv).sum(),
                                                                          "urban poor reached": t_q.loc[("area", "urban"), "recall_people_top20"],
                                                                          "rural poor reached": t_q.loc[("area", "rural"), "recall_people_top20"]}})
policy.to_csv(OUT / "poverty_area_cutoff_policy_option.csv")
policy

## 4. VUP coverage-gap model (holdout)

In [ ]:
SCORES_FINAL = ROOT / "data" / "processed" / "poverty_scores_final.parquet"
hn = vm.load_population(scores_path=SCORES_FINAL)
hdev = hn[hn["split"].eq("development")].reset_index(drop=True)
hhold = fr.add_groups(hn[hn["split"].eq("holdout")].reset_index(drop=True))
VX = hhold[vup_meta["feature_columns"]]
base_vup = cm.fit_pipeline(Pipeline([("model", cm.GroupRateBaseline("district"))]), hdev[["district"]], hdev["vup_any"], hdev["weight"])
hhold["district_baseline"] = base_vup.predict_proba(hhold[["district"]])[:, 1]
hhold["final"] = vup_final.predict_proba(VX)[:, 1]
VSCORES = ["district_baseline", "final"]
VLABEL = {"district_baseline": "District coverage baseline", "final": f"Deployment: {vup_meta['model']}"}
if isinstance(vup_final, fr.GroupRecalibratedClassifier):
    hhold["final_before_recalibration"] = vup_final.base.predict_proba(VX)[:, 1]
    VSCORES.append("final_before_recalibration")
    VLABEL["final_before_recalibration"] = "Same model before Platt recalibration"
print(f"High-need holdout: {len(hhold)} households, {int(hhold['vup_any'].sum())} VUP recipients")
VMETRICS = ["roc_auc", "pr_auc", "brier", "log_loss", "precision_top10", "recall_top10", "precision_top20", "recall_top20", "precision_top30", "recall_top30",
            "calibration_slope", "calibration_intercept", "calibration_in_the_large"]
vpoint = pd.DataFrame({s: cm.evaluate(hhold["vup_any"], hhold[s], hhold["weight"], hhold["pop_wt"]) for s in VSCORES}).loc[VMETRICS]
VB = ("roc_auc", "pr_auc", "brier", "precision_top10", "recall_top10", "precision_top20", "recall_top20", "precision_top30", "recall_top30", "calibration_in_the_large")
vb = cm.cluster_bootstrap(hhold, VSCORES, "vup_any", B=1000, baseline="district_baseline", metrics=VB).set_index(["score", "metric"])
vtab = vpoint.copy()
for s in VSCORES:
    vtab[f"{s} CI"] = [f"{vb.loc[(s, m), 'ci_low']:.3f}-{vb.loc[(s, m), 'ci_high']:.3f}" if (s, m) in vb.index else "" for m in VMETRICS]
vtab["final - baseline"] = [f"{vb.loc[('final', m), 'diff_vs_baseline_ci_low']:+.3f} to {vb.loc[('final', m), 'diff_vs_baseline_ci_high']:+.3f}" if ("final", m) in vb.index else "" for m in VMETRICS]
vtab.to_csv(OUT / "vup_holdout_metrics.csv")
vtab

In [ ]:
fig, axes = pl.figure("VUP coverage-gap model on the holdout",
                      f"High-need holdout households ({len(hhold)}; {int(hhold['vup_any'].sum())} recipients). Left: ROC; middle: precision-recall; right: calibration by quintile",
                      width=14, height=4.8, ncols=3)
yv_, wv_ = hhold["vup_any"], hhold["weight"]
VCOL = {"district_baseline": pl.NEUTRAL, "final": pl.SERIES[0], "final_before_recalibration": pl.SERIES[1]}
for s in VSCORES:
    fpr, tpr, _ = roc_curve(yv_, hhold[s], sample_weight=wv_)
    p_, r_, _ = precision_recall_curve(yv_, hhold[s], sample_weight=wv_)
    ls = "--" if s == "district_baseline" else ("-" if s == "final" else ":")
    axes[0].plot(fpr, tpr, ls, color=VCOL[s], linewidth=2, label=f"{VLABEL[s]} ({vpoint.loc['roc_auc', s]:.3f})")
    axes[1].plot(r_, p_, ls, color=VCOL[s], linewidth=2, label=f"{VLABEL[s]} ({vpoint.loc['pr_auc', s]:.3f})")
    c = cm.calibration_table(yv_, hhold[s], wv_, n_bins=5)
    axes[2].plot(c["mean_predicted"], c["observed"], "o-", color=VCOL[s], linewidth=2, label=VLABEL[s])
axes[0].plot([0, 1], [0, 1], ":", color=pl.BASELINE)
axes[1].axhline(np.average(yv_, weights=wv_), linestyle=":", color=pl.BASELINE)
axes[2].plot([0, 0.4], [0, 0.4], ":", color=pl.BASELINE)
for ax, xl, yl in zip(axes, ["False positive rate", "Recall", "Mean predicted coverage"], ["True positive rate", "Precision", "Observed coverage"]):
    ax.set_xlabel(xl, fontsize=9, color=pl.INK["secondary"]); ax.set_ylabel(yl, fontsize=9, color=pl.INK["secondary"])
    ax.grid(axis="both", color=pl.GRID); ax.legend(frameon=False, fontsize=7.5)
path = pl.save(fig, FIG / "vup_roc_pr_calibration.png", top=0.8, bottom=0.14, left=0.05, wspace=0.25)
display(Image(str(path)))
q = hhold.assign(qq=pd.qcut(hhold["final"].rank(method="first"), 5, labels=[f"Q{i}" for i in range(1, 6)]))
cov_q = cm.weighted_rate_table(q, "vup_any", "qq", "weight", [f"Q{i}" for i in range(1, 6)])
cov_q.to_csv(OUT / "vup_observed_coverage_by_predicted_quintile.csv")
print("Observed coverage by quintile of predicted coverage (Q1 = least likely covered):")
cov_q

In [ ]:
vsub = fr.group_table_ci(hhold, "final", "vup_any", B=500)
vsub_base = fr.group_table(hhold, "district_baseline", "vup_any")
vsub["district_baseline_roc_auc"] = vsub_base["roc_auc"].to_numpy()
vsub["district_baseline_mean_error"] = vsub_base["mean_error"].to_numpy()
vsub.to_csv(OUT / "vup_subgroups.csv", index=False)
print(f"Groups with >= 20 recipients (AUC reported): {int(vsub['reliable'].sum())} of {len(vsub)}")
display(pd.DataFrame({"final": fr.fairness_summary(vsub), "district baseline": fr.fairness_summary(vsub_base)}))
vsub[["dimension", "group", "households", "positives", "observed_rate", "mean_predicted", "mean_error", "mean_error_ci_low", "mean_error_ci_high",
      "roc_auc", "roc_auc_ci_low", "roc_auc_ci_high", "district_baseline_roc_auc", "precision_top20", "recall_people_top20"]]

## 5. Save

In [ ]:
rows = []
def add_rows(target, point_tab, boot_tab, scores, labels, roles, diff_boot=None, diff_name=None):
    for s in scores:
        for m in point_tab.index:
            r = {"model_target": target, "model": labels[s], "role": roles[s], "metric": m, "value": float(point_tab.loc[m, s])}
            if (s, m) in boot_tab.index:
                r.update({"ci_low": boot_tab.loc[(s, m), "ci_low"], "ci_high": boot_tab.loc[(s, m), "ci_high"],
                          "diff_vs_district_baseline_ci_low": boot_tab.loc[(s, m), "diff_vs_baseline_ci_low"],
                          "diff_vs_district_baseline_ci_high": boot_tab.loc[(s, m), "diff_vs_baseline_ci_high"]})
            if diff_boot is not None and (s, m) in diff_boot.index and s != diff_name:
                r.update({"diff_vs_unmitigated_ci_low": diff_boot.loc[(s, m), "diff_vs_baseline_ci_low"],
                          "diff_vs_unmitigated_ci_high": diff_boot.loc[(s, m), "diff_vs_baseline_ci_high"]})
            rows.append(r)

proles = {"district_baseline": "district_baseline", "unmitigated": "unmitigated" + (" (deployed)" if deploy == unm_name else ""),
          "recalibrated_base": "supplementary: recalibrated model's base",
          "recalibrated": "group_recalibrated" + (" (deployed)" if deploy == rec_name else "")}
add_rows("poverty", point, b_base, SCORES, {"district_baseline": "district_baseline", "unmitigated": unm_name, "recalibrated": rec_name,
                                           "recalibrated_base": rec_name.replace("+group_recal", "") + " (base of recalibrated)"}, proles, b_unm, "unmitigated")
vroles = {"district_baseline": "district_baseline", "final": "deployed", "final_before_recalibration": "deployed_before_recalibration"}
add_rows("vup_gap", vpoint, vb, VSCORES, {"district_baseline": "district_baseline", "final": vup_meta["model"], "final_before_recalibration": vup_meta["model"] + " (before Platt)"}, vroles)
for label, r in budget.iterrows():
    for k, v in r.items():
        rows.append({"model_target": "poverty", "model": label, "role": "budget_comparison", "metric": f"{k} at current VUP budget ({100 * k_vup:.1f}%)", "value": float(v)})
for s in ["district_baseline", "unmitigated", "recalibrated_base", "recalibrated"]:
    for k, v in fr.fairness_summary(sub[s]).items():
        rows.append({"model_target": "poverty", "model": LABEL[s], "role": proles[s], "metric": f"fairness_{k}", "value": float(v)})
for k, v in fr.fairness_summary(vsub).items():
    rows.append({"model_target": "vup_gap", "model": vup_meta["model"], "role": "deployed", "metric": f"fairness_{k}", "value": float(v)})
metrics = pd.DataFrame(rows)
metrics.to_csv(ROOT / "reports" / "final_model_metrics.csv", index=False)
print(f"reports/final_model_metrics.csv: {len(metrics)} rows")